# RAG Assignment: Baseline vs. Cross-Encoder Re-Ranking

**Document:** *CATAN – Game Rules & Almanac* (5th edition, 2020 print, 16 pages), the official rulebook published by Catan GmbH / Catan Studio.
It is a good stress test for retrieval because the same concept (trading, the robber, development cards…) is described in up to three
places (short Game Rules, alphabetical Almanac, back-cover Overview), and many rules have an exception or a "you may not…" clause that sits in a
different paragraph from the general rule.

| Component | Choice |
|---|---|
| PDF parsing | `pypdf` + custom cleaning |
| Chunking | sentence-aware, ≤ 90 words, 1-sentence overlap, never crosses a page |
| Bi-Encoder (Pipeline 1) | `sentence-transformers/all-MiniLM-L6-v2` (384-d), cosine similarity over L2-normalised vectors |
| Vector store | in-memory NumPy matrix (exact inner-product search) |
| Cross-Encoder (Pipeline 2) | `cross-encoder/ms-marco-MiniLM-L6-v2` |
| LLM | `Qwen/Qwen2.5-3B-Instruct` run locally with Hugging Face `transformers` (greedy decoding). Gemini can be used instead by setting `GEMINI_API_KEY`. |

Both pipelines share **the same chunks, the same bi-encoder top-10 candidate list and the same prompt**. The only difference is which 3 chunks reach the LLM.

## 1. Setup

In [ ]:
# pip install pypdf cryptography sentence-transformers transformers accelerate torch pandas
import os, re, time, logging, warnings
import numpy as np
import pandas as pd
import torch
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")
logging.getLogger("pypdf").setLevel(logging.ERROR)   # pypdf prints font warnings for this PDF
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

PDF_PATH = "data/catan_base_rules_2020.pdf"
BI_ENCODER = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER = "cross-encoder/ms-marco-MiniLM-L6-v2"
LOCAL_LLM = "Qwen/Qwen2.5-3B-Instruct"
GEMINI_MODEL = "gemini-2.5-flash"

TOP_K_RETRIEVE = 10   # candidates from the bi-encoder
TOP_K_CONTEXT = 3     # chunks given to the LLM

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
LLM_BACKEND = "gemini" if os.environ.get("GEMINI_API_KEY") else "local"
print("device:", DEVICE, "| LLM backend:", LLM_BACKEND)

## 2. Load and clean the PDF

The rulebook is a designed booklet, so raw extraction contains a lot of layout noise that would pollute the embeddings:
- `Y` glyphs that mark Almanac cross-references (e.g. `resources Y for`), and words split by them (`Y\nou` → `You`),
- small-caps headings extracted as `settin G`, `KniGht`, `lon Gest R oad`,
- number-token pips (`••••`), page numbers, `© 2015 Catan GmbH` captions, index dot-leaders,
- text printed on card illustrations (e.g. the victory-point card caption appears ~15 times).

These are removed with a few regular expressions; the rules text itself is not altered.

In [ ]:
# lines that are pure layout noise: blank, bullets, pips/numbers, doubled letters ("BB"), copyright, index leaders, figure labels
NOISE_LINE = re.compile(
    r"^(\s*|[•\s]+|[\d\s•:=]+|[A-Z]{2}(\s[A-Z]{2})*|©.*|\.?(\s*\.)*\s*\d*|"
    r"Illustration [A-Z]|(Robber|Harbor|Path|Intersection){2})$"
)

# small-caps headings are extracted with stray spaces/capitals
SPLIT_HEADINGS = {
    "Constru CtinG": "constructing", "settin G": "setting", "endin G": "ending",
    "Buildin G": "building", "distan Ce": "distance", "domesti C": "domestic",
    "interse Ctions": "intersections", "resour Ce": "resource", "produ Ction": "production",
    "rollin G": "rolling", "varia Ble": "variable", "almana C": "almanac",
    "startin G": "starting", "laRG est aRmy": "largest army", "lon Gest R oad": "longest road",
    "Odds fOr dice rOlls": "odds for dice rolls",
}


def clean_page(text):
    text = re.sub(r"\bY\s*\nou\b", "You", text)          # "Y\nou" -> "You"
    lines = [l.strip() for l in text.split("\n") if not NOISE_LINE.match(l.strip())]
    text = " ".join(lines)
    text = re.sub(r"\s+Y(?=[\s.,)”:])", "", text)        # Almanac cross-reference markers
    # captions printed on card artwork (repeated many times, not rules text)
    text = re.sub(r"1 Victory Point! Reveal this card on your turn if, with it, you reach the number "
                  r"of points required for victory\.( (LIBRARY|MARKET|CHAPEL|GREAT HALL|UNIVERSITY))?", "", text)
    text = re.sub(r"2 Victory Points! (This card goes to the player with the longest road of at least 5 "
                  r"segments\. Another player who builds a longer road takes this card\.|The first player "
                  r"to play 3 [Kk]night cards gets this card\. Another player who plays more [Kk]night cards "
                  r"takes this card\.)", "", text)
    for bad, good in SPLIT_HEADINGS.items():
        text = text.replace(bad, good)
    text = re.sub(r"\b[a-z]+[A-Z][A-Za-z]*\b", lambda m: m.group(0).lower(), text)  # "KniGht" -> "knight"
    return re.sub(r"\s+", " ", text).strip()


reader = PdfReader(PDF_PATH)
pages = [(i + 1, clean_page(p.extract_text() or "")) for i, p in enumerate(reader.pages)]
pages = [(n, t) for n, t in pages if len(t.split()) > 20]   # drops the cover page
print(f"{len(reader.pages)} pages in PDF, {len(pages)} with text, {sum(len(t.split()) for _, t in pages)} words")
print("\nSample (page 5):\n", pages[3][1][:700], "...")

## 3. Chunking

Sentence-aware chunks of at most ~90 words. Rules in this booklet are short, so small chunks keep one rule (or one exception) per chunk.
Each chunk repeats the previous chunk's last sentence for context (only if it is shorter than 40 words), and chunks never cross a page, so every chunk can be cited by page number.

In [ ]:
def chunk_pages(pages, max_words=90, overlap_sents=1):
    chunks = []
    for page_no, text in pages:
        sents = re.split(r"(?<=[.!?])\s+(?=[A-Z(“\"])", text)
        cur = []
        for s in sents:
            if cur and sum(len(x.split()) for x in cur) + len(s.split()) > max_words:
                chunks.append({"page": page_no, "text": " ".join(cur)})
                cur = cur[-overlap_sents:] if overlap_sents and len(cur[-1].split()) < 40 else []
            cur.append(s)
        if cur and len(" ".join(cur).split()) > 8:
            chunks.append({"page": page_no, "text": " ".join(cur)})
    for i, c in enumerate(chunks):
        c["id"] = i
    return chunks


chunks = chunk_pages(pages)
chunk_texts = [c["text"] for c in chunks]
lens = [len(t.split()) for t in chunk_texts]
print(f"{len(chunks)} chunks | words per chunk: mean {np.mean(lens):.0f}, min {min(lens)}, max {max(lens)}")
pd.DataFrame(chunks)[["id", "page", "text"]].head(8)

## 4. Pipeline 1 – Bi-Encoder index and baseline retrieval

Every chunk is embedded **once, offline** by the bi-encoder. Vectors are L2-normalised so the inner product equals cosine similarity.
At query time only the query is embedded, and a single matrix–vector product scores it against all chunks.

In [ ]:
bi_encoder = SentenceTransformer(BI_ENCODER, device=DEVICE)

t0 = time.perf_counter()
chunk_emb = bi_encoder.encode(chunk_texts, batch_size=32, normalize_embeddings=True, convert_to_numpy=True)
index_time = time.perf_counter() - t0
print(f"Indexed {chunk_emb.shape[0]} chunks -> matrix {chunk_emb.shape} in {index_time:.2f}s")


def retrieve(query, k=TOP_K_RETRIEVE):
    # Pipeline 1, step b: Top-k chunks by cosine similarity
    q = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True)[0]
    sims = chunk_emb @ q
    top = np.argsort(-sims)[:k]
    return [{"id": int(i), "page": chunks[i]["page"], "bi_rank": r + 1, "bi_score": float(sims[i]),
             "text": chunks[i]["text"]} for r, i in enumerate(top)]


def full_bi_rank(query, chunk_id):
    # rank of a chunk over the WHOLE corpus (used to diagnose recall misses outside the top 10)
    q = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True)[0]
    return int(np.where(np.argsort(-(chunk_emb @ q)) == chunk_id)[0][0]) + 1

## 5. Pipeline 2 – Cross-Encoder re-ranking

The cross-encoder reads `[CLS] query [SEP] chunk [SEP]` as **one** sequence, so every query token can attend to every chunk token.
It outputs a single relevance logit per pair. It is applied **only to the 10 bi-encoder candidates**.

In [ ]:
cross_encoder = CrossEncoder(CROSS_ENCODER, device=DEVICE)


def rerank(query, candidates):
    # Pipeline 2, steps b–c: score all (query, chunk) pairs, sort descending
    scores = cross_encoder.predict([(query, c["text"]) for c in candidates], batch_size=16)
    out = [dict(c, ce_score=float(s)) for c, s in zip(candidates, scores)]
    out.sort(key=lambda c: -c["ce_score"])
    for r, c in enumerate(out):
        c["ce_rank"] = r + 1
    return out

## 6. LLM and the shared prompt

The **same** prompt template is used for Answer 1 (bi-encoder top-3) and Answer 2 (re-ranked top-3). Decoding is greedy, so any
difference between the two answers comes from the context and not from sampling.

In [ ]:
PROMPT_TEMPLATE = """You are a rules assistant for the board game CATAN.
Answer the question using ONLY the rulebook excerpts below. If the excerpts do not contain the answer, say "The excerpts do not say."
Answer in at most 3 sentences and mention any exception that applies.

Rulebook excerpts:
{context}

Question: {question}
Answer:"""


def build_prompt(question, ctx_chunks):
    context = "\n\n".join(f"[Excerpt {i+1} | page {c['page']}]\n{c['text']}" for i, c in enumerate(ctx_chunks))
    return PROMPT_TEMPLATE.format(context=context, question=question)


if LLM_BACKEND == "local":
    tok = AutoTokenizer.from_pretrained(LOCAL_LLM)
    llm = AutoModelForCausalLM.from_pretrained(LOCAL_LLM, torch_dtype=torch.float16 if DEVICE != "cpu" else torch.float32).to(DEVICE)
    llm.eval()

    def generate(prompt, max_new_tokens=150):
        msgs = [{"role": "user", "content": prompt}]
        ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = llm.generate(ids, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None, top_k=None,
                               pad_token_id=tok.eos_token_id)
        return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()
else:
    from google import genai
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

    def generate(prompt, max_new_tokens=150):
        r = client.models.generate_content(model=GEMINI_MODEL, contents=prompt,
                                           config={"temperature": 0, "max_output_tokens": max_new_tokens})
        return r.text.strip()

print(generate("Reply with the single word: ready"))

## 7. The two end-to-end pipelines (with timing)

`time.perf_counter()` is recorded around each stage. Retrieval is shared, so Pipeline 2's retrieval cost is the baseline's cost plus the re-ranking cost.

In [ ]:
def run_both(query):
    t0 = time.perf_counter(); cands = retrieve(query); t_retrieve = time.perf_counter() - t0
    t0 = time.perf_counter(); reranked = rerank(query, cands); t_rerank = time.perf_counter() - t0

    top3_base, top3_rr = cands[:TOP_K_CONTEXT], reranked[:TOP_K_CONTEXT]
    t0 = time.perf_counter(); ans1 = generate(build_prompt(query, top3_base)); t_llm1 = time.perf_counter() - t0
    t0 = time.perf_counter(); ans2 = generate(build_prompt(query, top3_rr)); t_llm2 = time.perf_counter() - t0
    return {"query": query, "candidates": cands, "reranked": reranked, "answer1": ans1, "answer2": ans2,
            "t_retrieve": t_retrieve, "t_rerank": t_rerank, "t_llm1": t_llm1, "t_llm2": t_llm2}


def show(res, gold_phrases):
    # top-10 table: bi-encoder rank vs cross-encoder rank; * marks the chunk that actually answers the question
    rows = []
    for c in res["reranked"]:
        gold = any(g in c["text"].lower() for g in gold_phrases)
        rows.append({"chunk": c["id"], "page": c["page"], "bi_rank": c["bi_rank"], "bi_cos": round(c["bi_score"], 3),
                     "ce_rank": c["ce_rank"], "ce_score": round(c["ce_score"], 2), "gold": "*" if gold else "",
                     "text": c["text"][:110] + "…"})
    df = pd.DataFrame(rows).sort_values("bi_rank").set_index("bi_rank")
    print("QUERY:", res["query"])
    display(df)
    print("Baseline top-3 chunks:", [c["id"] for c in res["candidates"][:3]],
          "| Re-ranked top-3 chunks:", [c["id"] for c in res["reranked"][:3]])
    print("\nANSWER 1 (baseline):\n ", res["answer1"])
    print("\nANSWER 2 (re-ranked):\n ", res["answer2"])

## 8. Failure-mode stress tests

For every query a *gold* phrase identifies the chunk(s) that contain the deciding rule, so rank shifts are measured objectively rather than by eye.
Queries 1–5 are cases where the baseline misses or demotes the key chunk and re-ranking recovers it. Queries 6–7 are included on purpose
to show where re-ranking does **not** help.

In [ ]:
FAILURE_QUERIES = [
    {"type": "Negation & constraints",
     "query": "Is it prohibited to give resource cards to another player for free?",
     "gold": ["you may not give away cards", "cannot give away cards"]},
    {"type": "Exception / corner case",
     "query": "Which development card can I reveal on the same turn I buy it?",
     "gold": ["exception: if you buy a card"]},
    {"type": "Keyword trap / distractor",
     "query": "Can a settlement be built right next to my own settlement?",
     "gold": ["even yours"]},
    {"type": "Exception (rule variant)",
     "query": "Can I use a harbor on the same turn I build a settlement on it?",
     "gold": ["use a harbor on the same turn"]},
    {"type": "Buried chunk (ranks 4–10)",
     "query": "After playing Year of Plenty, can I spend those cards to build right away?",
     "gold": ["use these cards to build in the same turn"]},
    {"type": "Counter-example: re-ranker demotes the answer",
     "query": "Can I trade 2 wool for 1 wool?",
     "gold": ["trade like resources", "trade matching resources"]},
    {"type": "Counter-example: recall miss (not in top-10)",
     "query": "Am I allowed to leave the robber where it is after rolling a 7?",
     "gold": ["may not choose to leave the robber"]},
]

# warm-up so that one-off model initialisation is not counted in the latency numbers
_ = run_both("How many victory points do I need to win?")

results = []
for fq in FAILURE_QUERIES:
    results.append(run_both(fq["query"]))

In [ ]:
print("TYPE:", FAILURE_QUERIES[0]["type"])
show(results[0], FAILURE_QUERIES[0]["gold"])

In [ ]:
print("TYPE:", FAILURE_QUERIES[1]["type"])
show(results[1], FAILURE_QUERIES[1]["gold"])

In [ ]:
print("TYPE:", FAILURE_QUERIES[2]["type"])
show(results[2], FAILURE_QUERIES[2]["gold"])

In [ ]:
print("TYPE:", FAILURE_QUERIES[3]["type"])
show(results[3], FAILURE_QUERIES[3]["gold"])

In [ ]:
print("TYPE:", FAILURE_QUERIES[4]["type"])
show(results[4], FAILURE_QUERIES[4]["gold"])

In [ ]:
print("TYPE:", FAILURE_QUERIES[5]["type"])
show(results[5], FAILURE_QUERIES[5]["gold"])

In [ ]:
print("TYPE:", FAILURE_QUERIES[6]["type"])
show(results[6], FAILURE_QUERIES[6]["gold"])

## 9. Rank-shift summary

In [ ]:
def gold_rank(lst, gold, key):
    ranks = [c[key] for c in lst if any(g in c["text"].lower() for g in gold)]
    return min(ranks) if ranks else None

summary = []
for fq, res in zip(FAILURE_QUERIES, results):
    gold_ids = [c["id"] for c in chunks if any(g in c["text"].lower() for g in fq["gold"])]
    b = gold_rank(res["candidates"], fq["gold"], "bi_rank")
    r = gold_rank(res["reranked"], fq["gold"], "ce_rank")
    best_id = min(gold_ids, key=lambda i: full_bi_rank(fq["query"], i))
    summary.append({
        "type": fq["type"], "query": fq["query"], "gold chunk": best_id,
        "bi-encoder rank (full corpus)": full_bi_rank(fq["query"], best_id),
        "rank after re-ranking": r if r else "not in top-10",
        "in top-3: baseline": bool(b and b <= 3), "in top-3: re-ranked": bool(r and r <= 3),
    })
summary_df = pd.DataFrame(summary)
os.makedirs("results", exist_ok=True)
summary_df.to_csv("results/rank_shifts.csv", index=False)
summary_df

In [ ]:
# side-by-side answers for the report
answers_df = pd.DataFrame([{"type": fq["type"], "query": r["query"], "answer 1 (baseline)": r["answer1"],
                            "answer 2 (re-ranked)": r["answer2"]} for fq, r in zip(FAILURE_QUERIES, results)])
answers_df.to_csv("results/answers.csv", index=False)
with pd.option_context("display.max_colwidth", None):
    display(answers_df)

## 10. Latency profiling

Timings over a larger set of queries (the 7 stress tests plus 13 ordinary rules questions), after the warm-up.
LLM generation is timed separately because it is identical in cost for both pipelines (same prompt length, 3 chunks).

In [ ]:
EXTRA_QUERIES = [
    "How many victory points do I need to win?", "What does it cost to build a city?",
    "How many resource cards must I discard if I have 9 cards when a 7 is rolled?",
    "Who gets the Largest Army card?", "What is the distance rule?", "How does maritime trade work?",
    "Can I trade with the bank when it is not my turn?", "What does the Monopoly card do?",
    "How many roads, settlements and cities does each player have?", "Where does the robber start the game?",
    "How is the starting player chosen?", "What happens when the bank runs out of a resource?",
    "How many development cards can I play per turn?",
]
lat_queries = [fq["query"] for fq in FAILURE_QUERIES] + EXTRA_QUERIES

N_REPEAT = 5   # retrieval stages are fast, so repeat them for stable numbers
rows = []
for q in lat_queries:
    for _ in range(N_REPEAT):
        t0 = time.perf_counter(); cands = retrieve(q); t1 = time.perf_counter()
        rerank(q, cands); t2 = time.perf_counter()
        rows.append({"query": q, "bi_retrieve_ms": (t1 - t0) * 1e3, "ce_rerank_ms": (t2 - t1) * 1e3})
lat = pd.DataFrame(rows)

llm_times = [r["t_llm1"] for r in results] + [r["t_llm2"] for r in results]
stats = lambda s: {"mean": s.mean(), "median": s.median(), "p95": s.quantile(0.95)}
llm_ms = pd.Series(llm_times) * 1e3
latency_table = pd.DataFrame({
    "Pipeline 1 retrieval (embed query + top-10 search)": stats(lat.bi_retrieve_ms),
    "Cross-encoder re-rank of 10 pairs": stats(lat.ce_rerank_ms),
    "Pipeline 2 retrieval total (bi-encoder + re-rank)": stats(lat.bi_retrieve_ms + lat.ce_rerank_ms),
    "LLM generation (one answer, either pipeline)": stats(llm_ms),
}).T.round(1)
latency_table.columns = ["mean (ms)", "median (ms)", "p95 (ms)"]
latency_table.to_csv("results/latency.csv")
print(f"{len(lat_queries)} queries x {N_REPEAT} repeats on device={DEVICE}; one-off index build: {index_time*1e3:.0f} ms for {len(chunks)} chunks")
latency_table

In [ ]:
p1 = latency_table.loc["Pipeline 1 retrieval (embed query + top-10 search)", "mean (ms)"]
p2 = latency_table.loc["Pipeline 2 retrieval total (bi-encoder + re-rank)", "mean (ms)"]
llm_mean = latency_table.loc["LLM generation (one answer, either pipeline)", "mean (ms)"]
print(f"Retrieval only : re-ranking makes retrieval {p2/p1:.1f}x slower ({p1:.1f} ms -> {p2:.1f} ms)")
print(f"End-to-end     : {p1+llm_mean:.0f} ms vs {p2+llm_mean:.0f} ms  (+{100*(p2-p1)/(p1+llm_mean):.1f}%)")

## 11. Scaling experiment: why not run the cross-encoder over everything?

To make the scaling argument concrete, the cross-encoder is run over **all** chunks for one query and compared with the bi-encoder,
then both are extrapolated to larger collections. The bi-encoder's per-query cost is one query encoding plus a dot product per chunk
(chunk vectors are pre-computed). The cross-encoder needs a full transformer forward pass per (query, chunk) pair, and nothing can be pre-computed because each pass depends on the query.

In [ ]:
q = FAILURE_QUERIES[6]["query"]   # the recall-miss query
times_full = []
for _ in range(3):
    t0 = time.perf_counter()
    full_scores = cross_encoder.predict([(q, t) for t in chunk_texts], batch_size=32)
    times_full.append(time.perf_counter() - t0)
ce_full_ms = np.median(times_full) * 1e3
per_pair_ms = ce_full_ms / len(chunks)

times_bi = []
for _ in range(20):
    t0 = time.perf_counter(); retrieve(q); times_bi.append(time.perf_counter() - t0)
bi_ms = np.median(times_bi) * 1e3

order = np.argsort(-full_scores)
gold_id = [c["id"] for c in chunks if "may not choose to leave the robber" in c["text"].lower()][0]
print(f"Cross-encoder over all {len(chunks)} chunks: {ce_full_ms:.0f} ms ({per_pair_ms:.2f} ms/pair) vs bi-encoder retrieval {bi_ms:.1f} ms")
print(f"Gold chunk {gold_id}: bi-encoder rank {full_bi_rank(q, gold_id)} -> full-corpus cross-encoder rank {int(np.where(order == gold_id)[0][0]) + 1}")

scale = pd.DataFrame({"chunks in collection": [len(chunks), 10_000, 1_000_000, 100_000_000]})
scale["cross-encoder over all chunks"] = (scale["chunks in collection"] * per_pair_ms / 1e3).map(
    lambda s: f"{s:,.1f} s" if s < 3600 else f"{s/3600:,.1f} h")
scale["bi-encoder top-10 + re-rank 10"] = f"~{bi_ms + latency_table.loc['Cross-encoder re-rank of 10 pairs', 'median (ms)']:.0f} ms (+ ANN index lookup)"
scale.to_csv("results/scaling.csv", index=False)
scale

## 12. Observations (raw notes for the report)

*These are short factual notes taken from the outputs above. The written report is my own text.*

- See `results/rank_shifts.csv`, `results/answers.csv`, `results/latency.csv` and `results/scaling.csv` for the numbers used in the report.
- Queries 1–5: the deciding chunk was outside the baseline top-3 and the cross-encoder moved it into the top-3.
- Query 6: the cross-encoder pushed the correct chunk *out* of the top-3. Re-ranking is not always better.
- Query 7: the answer chunk is not in the bi-encoder top-10, so a re-ranker over 10 candidates cannot recover it. Recall is capped by the first stage.